In [ ]:
import pandas as pd
import os
import shutil
from pathlib import Path

## 1. Configuração de Diretórios

In [ ]:
# Diretórios
dir_tabelas = Path('relatorio/tabelas')
dir_img = Path('relatorio/img')

# Criar diretórios se não existirem
dir_tabelas.mkdir(parents=True, exist_ok=True)
dir_img.mkdir(parents=True, exist_ok=True)

print(f"Diretórios criados/verificados:")
print(f"  - {dir_tabelas}")
print(f"  - {dir_img}")

## 2. Funções Auxiliares

In [ ]:
def formatar_numero(valor):
    """Formata números para exibição em LaTeX."""
    if pd.isna(valor):
        return '-'
    
    if isinstance(valor, str):
        return valor.replace('_', '\\_')
    
    if isinstance(valor, bool):
        return 'Sim' if valor else 'Não'
    
    if isinstance(valor, (int, float)):
        if abs(valor) < 0.001 and valor != 0:
            return f"{valor:.2e}".replace('e-0', ' × 10$^{-').replace('e-', ' × 10$^{-') + '}$'
        elif isinstance(valor, float):
            return f"{valor:.4f}"
        else:
            return str(valor)
    
    return str(valor)

def traduzir_coluna(nome_coluna):
    """Traduz nomes de colunas para português."""
    traducoes = {
        'num_vertices': 'Vér.',
        'num_arestas': 'Are.',
        'densidade': 'Den.',
        'grau_medio': 'Grau Méd.',
        'grau_minimo': 'Grau Mín.',
        'grau_maximo': 'Grau Máx.',
        'eh_conexo': 'Conexo',
        'num_componentes': 'Comp.',
        'tamanho': 'Tam.',
        'instancia': 'Inst.',
        'id_instancia': 'ID',
        'id_grafo': 'ID',
        'num_cores': r'$\chi(G)$',
        'tempo_execucao': 'Tempo (s)',
        'tempo_segundos': 'Tempo (s)',
        'arestas': 'Are.',
        'instancia_id': 'Inst.',
        'caminho': 'Arquivo',
        'cores_heuristica': 'Cores',
        'cores_encontradas': 'Cores',
        'Algoritmo': 'Algoritmo',
        'Instâncias': 'Instâncias',
        'Cores Médias': 'Cores Méd.',
        'Cores Mín.': 'Cores Mín.',
        'Cores Máx.': 'Cores Máx.',
        'Tempo Médio (s)': 'Tempo Méd. (s)',
        'Qualidade': 'Qualidade',
        'Complexidade': 'Complexidade'
    }
    return traducoes.get(nome_coluna, nome_coluna.replace('_', ' ').title())

def gerar_tabela_latex(df, titulo, label, colunas_remover=None, max_linhas=None, fonte_pequena=False):
    """Gera código LaTeX para uma tabela a partir de um DataFrame."""
    
    if colunas_remover:
        df = df.drop(columns=[col for col in colunas_remover if col in df.columns], errors='ignore')
    
    if max_linhas and len(df) > max_linhas:
        df = df.head(max_linhas)
        nota_truncada = f"\\multicolumn{{{len(df.columns)}}}{{c}}{{\\textit{{... {len(df)} primeiras linhas ...}}}} \\\\"
    else:
        nota_truncada = ""
    
    colunas_traduzidas = [traduzir_coluna(col) for col in df.columns]
    
    df_formatado = df.copy()
    for col in df_formatado.columns:
        df_formatado[col] = df_formatado[col].apply(formatar_numero)
    
    num_cols = len(df.columns)
    alinhamento = 'c' * num_cols
    
    fonte_cmd = "\\footnotesize\n" if fonte_pequena else ""
    
    latex_code = f"""% Tabela gerada automaticamente - Resultados 1
\\begin{{table}}[H]
\\centering
\\caption{{{titulo}}}
\\label{{{label}}}
{fonte_cmd}\\begin{{tabular}}{{{alinhamento}}}
\\toprule
"""
    
    latex_code += " & ".join([f"\\textbf{{{col}}}" for col in colunas_traduzidas]) + " \\\\\n"
    latex_code += "\\midrule\n"
    
    for _, row in df_formatado.iterrows():
        latex_code += " & ".join([str(val) for val in row.values]) + " \\\\\n"
    
    if nota_truncada:
        latex_code += nota_truncada + "\n"
    
    latex_code += """\\bottomrule
\\end{tabular}
\\end{table}
"""
    
    return latex_code

print("✓ Funções auxiliares definidas")

## 3. Parte 1 - Força Bruta

### 3.1 Parâmetros dos Grafos

In [ ]:
df_parametros = pd.read_csv('resultados_1/parte1/csv/parametros_grafos.csv')

print(f"Parâmetros dos Grafos: {len(df_parametros)} linhas")
print(f"Colunas: {list(df_parametros.columns)}")
display(df_parametros.head(10))

latex_parametros = gerar_tabela_latex(
    df_parametros,
    titulo="Parâmetros Estruturais dos Grafos - Resultados 1 (Parte 1)",
    label="tab:parametros_grafos_r1",
    colunas_remover=['id_grafo', 'id_instancia'],
    max_linhas=15,
    fonte_pequena=True
)

arquivo_tex = dir_tabelas / 'tabela_parametros_grafos_r1.tex'
with open(arquivo_tex, 'w', encoding='utf-8') as f:
    f.write(latex_parametros)

print(f"\n✓ Tabela salva em: {arquivo_tex}")

### 3.2 Resultados Força Bruta

In [ ]:
df_forca_bruta = pd.read_csv('resultados_1/parte1/csv/resultados_forca_bruta.csv')

print(f"Resultados Força Bruta: {len(df_forca_bruta)} linhas")
print(f"Colunas: {list(df_forca_bruta.columns)}")
display(df_forca_bruta.head(10))

latex_forca_bruta = gerar_tabela_latex(
    df_forca_bruta,
    titulo="Resultados do Algoritmo de Força Bruta - Resultados 1 (Parte 1)",
    label="tab:resultados_forca_bruta_r1",
    colunas_remover=['id_grafo', 'id_instancia'],
    max_linhas=15
)

arquivo_tex = dir_tabelas / 'tabela_resultados_forca_bruta_r1.tex'
with open(arquivo_tex, 'w', encoding='utf-8') as f:
    f.write(latex_forca_bruta)

print(f"\n✓ Tabela salva em: {arquivo_tex}")

## 4. Parte 2 - Heurísticas (Welsh-Powell + DSATUR)

### 4.1 Informações das Instâncias DIMACS

In [ ]:
df_info = pd.read_csv('resultados_1/parte2/csv/informacoes_instancias.csv')

print(f"Informações Instâncias: {len(df_info)} linhas")
print(f"Colunas: {list(df_info.columns)}")
display(df_info)

latex_info = gerar_tabela_latex(
    df_info,
    titulo="Informações das Instâncias DIMACS - Resultados 1 (Parte 2)",
    label="tab:informacoes_instancias_r1",
    fonte_pequena=True
)

arquivo_tex = dir_tabelas / 'tabela_informacoes_instancias_r1.tex'
with open(arquivo_tex, 'w', encoding='utf-8') as f:
    f.write(latex_info)

print(f"\n✓ Tabela salva em: {arquivo_tex}")

### 4.2 Resultados Welsh-Powell

In [ ]:
df_wp = pd.read_csv('resultados_1/parte2/csv/heuristica_Welsh-Powell.csv')

print(f"Resultados Welsh-Powell: {len(df_wp)} linhas")
print(f"Colunas: {list(df_wp.columns)}")
display(df_wp)

latex_wp = gerar_tabela_latex(
    df_wp,
    titulo="Resultados da Heurística Welsh-Powell - Resultados 1 (Parte 2)",
    label="tab:welsh_powell_r1",
    colunas_remover=['caminho'],
    fonte_pequena=True
)

arquivo_tex = dir_tabelas / 'tabela_welsh_powell_r1.tex'
with open(arquivo_tex, 'w', encoding='utf-8') as f:
    f.write(latex_wp)

print(f"\n✓ Tabela salva em: {arquivo_tex}")

### 4.3 Resultados DSATUR

In [ ]:
df_dsatur = pd.read_csv('resultados_1/parte2/csv/heuristica_DSATUR.csv')

print(f"Resultados DSATUR: {len(df_dsatur)} linhas")
print(f"Colunas: {list(df_dsatur.columns)}")
display(df_dsatur)

latex_dsatur = gerar_tabela_latex(
    df_dsatur,
    titulo="Resultados da Heurística DSATUR - Resultados 1 (Parte 2)",
    label="tab:dsatur_r1",
    colunas_remover=['caminho'],
    fonte_pequena=True
)

arquivo_tex = dir_tabelas / 'tabela_dsatur_r1.tex'
with open(arquivo_tex, 'w', encoding='utf-8') as f:
    f.write(latex_dsatur)

print(f"\n✓ Tabela salva em: {arquivo_tex}")

## 5. Comparação entre os 3 Algoritmos

In [ ]:
df_comparacao = pd.read_csv('resultados_1/comparacao_algoritmos.csv')

print(f"Comparação: {len(df_comparacao)} linhas")
print(f"Colunas: {list(df_comparacao.columns)}")
display(df_comparacao)

latex_comparacao = gerar_tabela_latex(
    df_comparacao,
    titulo="Comparação entre os 3 Algoritmos - Resultados 1",
    label="tab:comparacao_algoritmos_r1"
)

arquivo_tex = dir_tabelas / 'tabela_comparacao_algoritmos_r1.tex'
with open(arquivo_tex, 'w', encoding='utf-8') as f:
    f.write(latex_comparacao)

print(f"\n✓ Tabela salva em: {arquivo_tex}")

## 6. Resumo

In [ ]:
print("="*80)
print("RESUMO - TABELAS LATEX GERADAS (RESULTADOS 1)")
print("="*80)
print(f"\n📊 Parte 1 - Força Bruta:")
print(f"  ✓ tabela_parametros_grafos_r1.tex")
print(f"  ✓ tabela_resultados_forca_bruta_r1.tex")
print(f"\n📊 Parte 2 - Heurísticas:")
print(f"  ✓ tabela_informacoes_instancias_r1.tex")
print(f"  ✓ tabela_welsh_powell_r1.tex")
print(f"  ✓ tabela_dsatur_r1.tex")
print(f"\n📊 Comparação:")
print(f"  ✓ tabela_comparacao_algoritmos_r1.tex")
print(f"\n📁 Localização: {dir_tabelas}")
print(f"\n✅ Total: 6 tabelas LaTeX geradas com sucesso!")
print("="*80)